In [1]:
import os
from datetime import datetime

import pandas as pd
from tqdm import tqdm

import wandb

In [2]:
%cd "/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis/"

/pfs/work9/workspace/scratch/ma_faroesch-master-thesis/playground-uc3/master-thesis


In [3]:
# Konfiguration
CSV_PATH = "test_grid_experiments_extended.csv"
WANDB_ENTITY = "roesch01-university-of-mannheim"  # Falls nötig, hier deinen Usernamen/Teamnamen eintragen
WANDB_PROJECT = "master-thesis-extended"  # Falls nötig, hier den Projektnamen eintragen
TAG_FILTER = "Final"

date_threshold = datetime(year=2026, month=2, day=5)

In [22]:
def update_csv_with_wandb_ids():
    # 1. CSV laden
    if not os.path.exists(CSV_PATH):
        raise FileNotFoundError(f"Die Datei {CSV_PATH} wurde nicht gefunden.")
    
    df = pd.read_csv(CSV_PATH, sep=';', index_col=None)
    
    # Spalte für ID vorbereiten, falls noch nicht da
    df['wandb_run_id'] = None

    # 2. W&B API initialisieren
    api = wandb.Api()

    print(f"Starte Abgleich für {len(df)} Zeilen...")

    # 3. Iteration mit Fortschrittsbalken
    pbar = tqdm(df.iterrows(), total=len(df), desc="Matching Runs")
    for index, row in pbar:

        if index > 65:
            break
        
        # Filter-Kriterien aus der CSV Zeile
        # Hinweis: Wir casten lambda_epg auf float, falls es als String geladen wurde
        concept_module = row['concept_module_name']
        dataset = row['dataset']
        segmentation_module = row['segmentation_module_name']
        use_soft_labels = not pd.isna(row['use_soft_labels'])
        concept_criterion = row['concept_criterion']
        mask_reg_criterion = row['mask_reg_criterion']
        lambda_mask_reg_loss = row['lambda_mask_reg_loss']
        affinity_criterion = row['affinity_criterion']
        lambda_affinity_loss = row['lambda_affinity_loss']
        top_k_percent = row['top_k_percent']
        mask_criterion = row['mask_criterion']
        classification_criterion = row['classification_criterion']
        n_concept_implicitely_learned = row['n_concept_implicitely_learned']
        lambda_concept_reg = row['lambda_concept_reg_loss']
        
        pbar.set_postfix({"concept_module": concept_module, "segmentation_module": segmentation_module, "dataset": dataset})
        
        # W&B Query (MongoDB Syntax)
        # Wir filtern nach Projekt, Tag und den drei spezifischen Config-Parametern
        filters = {
            "tags": {"$in": [TAG_FILTER]},
            "config.dataset": dataset,
            "config.architecture.concept_module": concept_module,
            "config.use_soft_labels": use_soft_labels,
            "created_at": {"$gt": date_threshold.isoformat()}
        }
        if pd.isna(segmentation_module):
            filters["config.architecture.unified_model"] = row['unified_name']
        else:
            filters["config.architecture.segmentation_module"] = segmentation_module

        if not pd.isna(concept_criterion):
            filters["config.criteria.concept"] = concept_criterion

        if not pd.isna(mask_reg_criterion):
            filters["config.criteria.mask_reg"] = mask_reg_criterion
        else:
            filters["config.criteria.mask_reg"] = None

        if not pd.isna(lambda_mask_reg_loss):
            filters["config.loss_weights.lambda_mask_reg"] = lambda_mask_reg_loss
        else:            
            filters["config.loss_weights.lambda_mask_reg"] = None

        if not pd.isna(affinity_criterion):
            filters["config.criteria.affinity"] = affinity_criterion
        else:
            filters["config.criteria.affinity"] = None

        if not pd.isna(mask_criterion):
            filters["config.criteria.mask"] = mask_criterion
        else:
            filters["config.criteria.mask"] = None

        if not pd.isna(lambda_affinity_loss):
            filters["config.loss_weights.lambda_affinity"] = lambda_affinity_loss
        else:
            filters["config.loss_weights.lambda_affinity"] = None

        if not pd.isna(lambda_concept_reg):
            filters["config.loss_weights.lambda_concept_reg"] = lambda_concept_reg
        else:
            filters["config.loss_weights.lambda_concept_reg"] = None

        if not pd.isna(top_k_percent):
            filters["config.hyperparameters.top_k_percent"] = top_k_percent
        else:
            filters["config.hyperparameters.top_k_percent"] = None

        if not pd.isna(classification_criterion):
            filters["config.criteria.classification"] = classification_criterion
        else:
            filters["config.criteria.classification"] = None



        
        runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT, filters=filters)
        
        
        try:
            run_list = list(runs)

        except Exception as e:
            filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
            print(f"FEHLER bei W&B API Anfrage für Zeile {index}: {e}")
            print(f"Filter: {filters_readable}")
            raise

        # 4. Validierung & Fehlerbehandlung
        if len(run_list) == 0:
            filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
            raise ValueError(
                f"FEHLER: Kein Run gefunden für:\n"
                f"Filter: {filters_readable}"
            )
        
        if len(run_list) > 1:

            run_found = False

            for run in runs:
                cmd = " ".join(run.metadata.get("args", []))

                # print(f"--n-concepts-implicitly-learned {int(n_concept_implicitely_learned)} ", cmd)
                
                if f"--n-concepts-implicitly-learned {int(n_concept_implicitely_learned)} " in cmd:
                    run_found = True
                    run_list = [run]
                    break


            if not run_found:
                run_ids = [r.id for r in run_list]
                filters_readable = '\n'.join([f'{k}: {v}' for k, v in filters.items()])
                raise ValueError(
                    f"FEHLER: Mehrere Runs ({len(run_list)}) gefunden für:\n"
                    f"Filter: {filters_readable}\n"
                    f"IDs: {run_ids}"
                )
            
            

        run = run_list[0]

        mask = df["wandb_run_id"] == run.id

        if mask.any():
            print(df.loc[mask])
            raise ValueError(f"Run {run.id} existiert bereits im DataFrame.")
        
        # ID eintragen
        df.at[index, 'wandb_run_id'] = run_list[0].id

    return df

In [23]:
data = update_csv_with_wandb_ids()

Starte Abgleich für 397 Zeilen...


Matching Runs:  17%|█▋        | 66/397 [00:40<03:23,  1.62it/s, concept_module=LogitMeanTopK, segmentation_module=SegmentationHeadUpscaledMulti, dataset=FunnyBirds]                    


In [24]:
data.columns

Index(['test_id', 'dataset', 'lr', 'epochs', 'dino_ckpt_segdino',
       'unified_name', 'encoder_name', 'upsampler_name',
       'segmentation_module_name', 'concept_module_name',
       'classification_module_name', 'freeze_encoder', 'freeze_upsampler',
       'mask_criterion', 'affinity_criterion', 'affinity_sim_threshold',
       'tv_criterion', 'concept_criterion', 'classification_criterion',
       'mask_reg_criterion', 'concept_reg_criterion',
       'lambda_concept_reg_loss', 'lambda_tv_loss', 'lambda_mask_reg_loss',
       'lambda_affinity_loss', 'lambda_mask_loss', 'lambda_concept_loss',
       'lambda_concept_reg_loss.1', 'lambda_classification_loss',
       'calculate_dice', 'calculate_fg_dice', 'calculate_iou', 'batch_size',
       'img_size', 'top_k_percent', 'n_concept_implicitely_learned',
       'use_soft_labels', 'attr_level', 'wandb_run_id', 'Unnamed: 36'],
      dtype='object')

In [25]:
data = data.drop(columns=['Unnamed: 36'])

In [26]:
data['test_id'] = data['test_id'].astype(int)
data['epochs'] = data['epochs'].astype(int)
data['batch_size'] = data['batch_size'].astype(int)
data['img_size'] = data['img_size'].astype(int)
data

,test_id,dataset,lr,epochs,dino_ckpt_segdino,unified_name,encoder_name,upsampler_name,segmentation_module_name,concept_module_name,...,calculate_dice,calculate_fg_dice,calculate_iou,batch_size,img_size,top_k_percent,n_concept_implicitely_learned,use_soft_labels,attr_level,wandb_run_id
0,0,FunnyBirds,0.0001,25,NaN,NaN,dinov3,anyup,SegmentationHeadUpscaledSingle,SegMaskAvgPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,i6n2g0yg
1,1,FunnyBirds,0.0001,25,NaN,NaN,dinov3,anyup,SegmentationHeadUpscaledMulti,SegMaskAvgPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,c01q4i6o
2,2,FunnyBirds,0.0001,25,NaN,NaN,dinov3,NaN,SegmentationHeadSETRPUP,SegMaskAvgPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,xlp9yh1g
3,3,FunnyBirds,0.0001,25,dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth,segdino_b,NaN,NaN,NaN,SegMaskAvgPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,1jpajf8d
4,4,CUB_112,0.0001,100,NaN,NaN,dinov3,anyup,SegmentationHeadUpscaledSingle,SegMaskAvgPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,5bhtbcxc
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
392,331,CUB_112,0.0001,200,dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth,segdino_b,NaN,NaN,NaN,SegMaskWeightedPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,None
393,332,CUB_112,0.0001,200,NaN,NaN,dinov3,anyup,SegmentationHeadSingleLayer,SegFeatureWeightedSoftmaxPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,None
394,333,CUB_112,0.0001,200,NaN,NaN,dinov3,anyup,SegmentationHeadDoubleLayer,SegFeatureWeightedSoftmaxPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,None
395,334,CUB_112,0.0001,200,NaN,NaN,dinov3,anyup,SegmentationHeadRefinement,SegFeatureWeightedSoftmaxPool,...,True,True,True,4,256,NaN,NaN,NaN,NaN,None


In [ ]:
DATA_PATH = 'thesis-figures/extended_cbm/results/0_extended_cbm_results.parquet'

# 5. Speichern
data.to_csv(CSV_PATH, sep=';', index=False)
print(f"\nErfolg! Die CSV wurde aktualisiert und gespeichert unter: {CSV_PATH}")


Erfolg! Die CSV wurde aktualisiert und gespeichert unter: test_grid_experiments_extended.csv


: 

In [23]:
data['wandb_run_id'].iloc[:54]

0     i6n2g0yg
1     c01q4i6o
2     xlp9yh1g
3     1jpajf8d
4     5bhtbcxc
5     av3igr7l
6     q0gevqso
7     pbvhti5j
8     e2xe2ikz
9     n5zozke9
10    wzcsobfy
11    oxvomshq
12    rypsd8q1
13    pde5gcnk
14    v4n2uon6
15    vzs2vei9
16    dwxi34vh
17    d8oxb1yg
18    jguzfmll
19    n1ohshjk
20    3vardl0g
21    n4lrcx2q
22    8s5i66xe
23    k35hft9t
24    u4w58p5i
25    48x2ek26
26    652znd0o
27    dstfi0h2
28    8jefa23q
29    mn106ezy
30    uxcl3qvy
31    nazemmyd
32    gv70wmzz
33    frtwr5la
34    ol2s9hbj
35    w9mn773s
36    ecb3hwum
37    cmxceof3
38    xkfv5cvz
39    g3c6b6bf
40    qatqo9db
41    sltpp8ev
42    quos7n74
43    17m6p70g
44    aq81keiy
45    9lexo76a
46    c0ldpb7v
47    msu693b8
48    2effny07
49    9i6fpi2h
50    ij9iskh2
51    tidjt3zg
52    ewzmye18
53    k9anbolk
Name: wandb_run_id, dtype: object